In [12]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from pathlib import Path

SEED = 42
DECAY_GAMMA = 0.03
EB_KAPPA = 500

# Paths
TX_PATH = "../data/curated/merchant_transactions"
PROB_PATH = "../data/tables/merchant_data/consumer_fraud_probability.csv"
OUTPUT_DIR = "../artifacts/fraud_outputs"
MERCHANTS_PATH = "../data/tables/merchant_data/tbl_merchants.parquet"
CURATED_DIR = "../data/curated"

conf = (
    SparkConf()
    .setAppName("fraud_prob_pipeline")
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .set("spark.sql.shuffle.partitions", "200")
    .set("spark.driver.memory", "6g")
    .set("spark.executor.memory", "6g")
)

spark = (
    SparkSession.builder.master("local[*]").config(conf=conf).getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [13]:
# rwad in transaction data and print column names

tx_df = spark.read.parquet(TX_PATH)
tx_df.printSchema()

root
 |-- merchant_abn: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_datetime: date (nullable = true)
 |-- business: string (nullable = true)
 |-- biz_tags: string (nullable = true)
 |-- rev_band: string (nullable = true)
 |-- take_rate: string (nullable = true)
 |-- segment: string (nullable = true)



### Pipeline overview
- Load transactions and direct fraud probabilities
- Clean transactions
- Tiered probability: direct match (Tier A), user-decay imputation (Tier B), model-based imputation (Tier C)
- Aggregate to merchant-level with empirical Bayes shrinkage and compute FraudScore
- Report merchant rankings


In [14]:
# Global tunables
LAMBDA = 1.0  # EB/fraud score adjustment strength


### Load & Basic Hygiene


In [15]:
# Read inputs

# Define schemas for strict typing
schema_tx = T.StructType([
    T.StructField("merchant_abn", T.LongType(), True),
    T.StructField("user_id", T.LongType(), True),
    T.StructField("dollar_value", T.DoubleType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("business", T.StringType(), True),
    T.StructField("biz_tags", T.StringType(), True),
    T.StructField("rev_band", T.StringType(), True),
    T.StructField("take_rate", T.StringType(), True),
    T.StructField("segment", T.StringType(), True),
])

schema_prob = T.StructType([
    T.StructField("user_id", T.LongType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("fraud_probability", T.DoubleType(), True),
])

# Load primary inputs
transactions = spark.read.schema(schema_tx).parquet(TX_PATH)
probs = spark.read.option("header", True).schema(schema_prob).csv(PROB_PATH)

# Normalize probabilities to [0,1] if given as percent
probs = probs.withColumn(
    "fraud_probability",
    F.when(F.col("fraud_probability") > 1.0, F.col("fraud_probability") / F.lit(100.0)).otherwise(F.col("fraud_probability"))
)

# Text hygiene & parse merchant take_rate to numeric
transactions = (
    transactions
    .withColumn("biz_tags", F.trim(F.regexp_replace(F.col("biz_tags"), "\s+", " ")))
    .withColumn("rev_band", F.trim(F.col("rev_band")))
    .withColumn("take_rate", F.trim(F.col("take_rate")))
    .withColumn("segment", F.trim(F.col("segment")))
    .withColumn(
        "take_rate_num",
        F.when(F.col("take_rate").rlike(r"^[0-9.]+%$"), F.regexp_replace("take_rate", "%", "").cast("double") / 100.0)
         .when(F.col("take_rate").rlike(r"^[0-9.]+$"), F.col("take_rate").cast("double"))
         .otherwise(F.lit(None).cast("double"))
    )
)

# Cache & counts
transactions.cache(); probs.cache()
print("Transactions:", transactions.count())
print("Probs rows:", probs.count())


25/10/09 21:03:58 WARN CacheManager: Asked to cache already cached data.


Transactions: 13293853
Probs rows: 34864


In [16]:
transactions.show(20)
probs.show(20)

+------------+-------+------------------+--------------+--------------------+--------------------+--------+---------+--------------------+-------------+
|merchant_abn|user_id|      dollar_value|order_datetime|            business|            biz_tags|rev_band|take_rate|             segment|take_rate_num|
+------------+-------+------------------+--------------+--------------------+--------------------+--------+---------+--------------------+-------------+
| 70620117107|  18491| 135.5756236214744|    2021-08-20|Ante Vivamus Cons...|bicycle shops - s...|       a|     6.06|Lifestyle, Health...|         6.06|
| 70620117107|   9538|159.47053512730272|    2022-03-02|Ante Vivamus Cons...|bicycle shops - s...|       a|     6.06|Lifestyle, Health...|         6.06|
| 40099570304|  18518| 91.51349059851854|    2021-08-20|           Nulla LLC|bicycle shops - s...|       c|     3.10|Lifestyle, Health...|          3.1|
| 79633007926|   9612|11.560651626732605|    2022-03-02|   Auctor Vitae Inc.|bicyc

### Tier A: Direct probability matches


In [17]:
# Left join on (user_id, order_datetime) to get p_direct

transactions = transactions.withColumn("order_date", F.col("order_datetime"))
probs = probs.withColumn("order_date", F.col("order_datetime")).drop("order_datetime")

joined_a = transactions.join(
    probs.select("user_id", "order_date", F.col("fraud_probability").alias("p_direct")),
    ["user_id", "order_date"],
    "left",
)

print("Tier A: p_direct non-null:", joined_a.filter(F.col("p_direct").isNotNull()).count())
joined_a.cache()


Tier A: p_direct non-null: 54764


DataFrame[user_id: bigint, order_date: date, merchant_abn: bigint, dollar_value: double, order_datetime: date, business: string, biz_tags: string, rev_band: string, take_rate: string, segment: string, take_rate_num: double, p_direct: double]

In [18]:
# How many distinct user-day keys in probs?
probs_pairs = probs.select("user_id", "order_date").distinct().count()

# How many transaction rows got a direct match?
matches_rows = joined_a.filter(F.col("p_direct").isNotNull()).count()

# How many distinct user-day pairs among those matches?
matches_pairs = (
    joined_a.filter(F.col("p_direct").isNotNull())
            .select("user_id", "order_date")
            .distinct()
            .count()
)

print("probs distinct pairs:", probs_pairs)
print("matched rows:", matches_rows)
print("matched distinct pairs:", matches_pairs)
print("avg tx per matched pair:", matches_rows / matches_pairs)

probs distinct pairs: 34765
matched rows: 54764
matched distinct pairs: 28010
avg tx per matched pair: 1.9551588718314887


In [19]:
from pyspark.sql import functions as F

num_users = transactions.select("user_id").where(
    F.col("user_id").isNotNull()).distinct().count()

num_transactions = transactions.select()
num_merchants = transactions.select("merchant_abn").where(
    F.col("merchant_abn").isNotNull()).distinct().count()

print(f"Distinct users (transactions): {num_users}")
print(f"Distinct merchants (transactions): {num_merchants}")

Distinct users (transactions): 24081
Distinct merchants (transactions): 3925


In [20]:
from pyspark.sql import functions as F

# All merchants seen in transactions
merchants_all = (
    transactions.select("merchant_abn")
    .where(F.col("merchant_abn").isNotNull())
    .distinct()
)

# Merchants with at least one p_direct
merchants_with_p = (
    joined_a.where(F.col("p_direct").isNotNull())
            .select("merchant_abn")
            .where(F.col("merchant_abn").isNotNull())
            .distinct()
)

# Merchants with no p_direct at all (anti-join)
merchants_no_p = merchants_all.join(
    merchants_with_p, "merchant_abn", "left_anti")

print("Total merchants:", merchants_all.count())
print("Merchants with any p_direct:", merchants_with_p.count())
print("Merchants with no p_direct at all:", merchants_no_p.count())

Total merchants: 3925
Merchants with any p_direct: 2789
Merchants with no p_direct at all: 1136


In [21]:
from pyspark.sql import functions as F

# Reuse the same merchant sets you derived
merchants_all = (
    transactions.select("merchant_abn")
    .where(F.col("merchant_abn").isNotNull())
    .distinct()
)

merchants_with_p = (
    joined_a.where(F.col("p_direct").isNotNull())
            .select("merchant_abn")
            .where(F.col("merchant_abn").isNotNull())
            .distinct()
)

merchants_no_p = merchants_all.join(
    merchants_with_p, "merchant_abn", "left_anti")

# Transaction counts per group (restrict to non-null merchant_abn for consistency)
tx_nonnull_merch = transactions.where(
    F.col("merchant_abn").isNotNull()).count()
tx_with_p_merchants = transactions.join(F.broadcast(
    merchants_with_p), "merchant_abn", "inner").count()
tx_no_p_merchants = transactions.join(F.broadcast(
    merchants_no_p), "merchant_abn", "inner").count()

print(
    f"Transactions with merchants having any p_direct: {tx_with_p_merchants}")
print(
    f"Transactions with merchants having no p_direct at all: {tx_no_p_merchants}")
print(f"Total transactions with non-null merchant_abn: {tx_nonnull_merch}")
print(
    f"Sanity check (sum matches): {tx_with_p_merchants + tx_no_p_merchants == tx_nonnull_merch}")

# Optional shares
print(f"Share (any p_direct): {tx_with_p_merchants/tx_nonnull_merch:.2%}")
print(f"Share (no p_direct):  {tx_no_p_merchants/tx_nonnull_merch:.2%}")

Transactions with merchants having any p_direct: 13144511
Transactions with merchants having no p_direct at all: 149342
Total transactions with non-null merchant_abn: 13293853
Sanity check (sum matches): True
Share (any p_direct): 98.88%
Share (no p_direct):  1.12%


### Tier B: User-level propensity with time decay


In [22]:
# Compute p_user_decay for rows without p_direct

# Prepare per-user probability history as (user_id, d_k, p_k)
prob_hist = probs.select(
    "user_id", F.col("order_date").alias("d_k"), F.col("fraud_probability").alias("p_k")
)

# For efficiency: join only for users present in transactions lacking p_direct
users_needing = joined_a.filter(F.col("p_direct").isNull()).select("user_id").distinct()

cand = (
    joined_a.select("user_id", "order_date")
    .join(users_needing, "user_id", "inner")
    .join(prob_hist, "user_id", "inner")
)

# Compute weights w_k = exp(-gamma * |t - d_k|) in days
diff_days = F.abs(F.datediff(F.col("order_date"), F.col("d_k")))
cand = cand.withColumn("w_k", F.exp(-DECAY_GAMMA * diff_days))

# Aggregate per (user_id, order_date)
p_user_decay_df = (
    cand.groupBy("user_id", "order_date")
    .agg(
        (F.sum(F.col("p_k") * F.col("w_k")) / F.sum(F.col("w_k"))).alias("p_user_decay")
    )
)

# Join back onto joined_a, only where p_direct is null
joined_b = (
    joined_a
    .join(p_user_decay_df, ["user_id", "order_date"], "left")
    .withColumn("p_user_decay", F.when(F.col("p_direct").isNull(), F.col("p_user_decay")).otherwise(F.lit(None)))
)

print("Tier B: filled via decay:", joined_b.filter(F.col("p_user_decay").isNotNull()).count())
joined_b.cache()


Tier B: filled via decay: 11056382


DataFrame[user_id: bigint, order_date: date, merchant_abn: bigint, dollar_value: double, order_datetime: date, business: string, biz_tags: string, rev_band: string, take_rate: string, segment: string, take_rate_num: double, p_direct: double, p_user_decay: double]

In [23]:
joined_b.show(20)

+-------+----------+------------+------------------+--------------+--------------------+--------------------+--------+---------+--------------------+-------------+--------+-------------------+
|user_id|order_date|merchant_abn|      dollar_value|order_datetime|            business|            biz_tags|rev_band|take_rate|             segment|take_rate_num|p_direct|       p_user_decay|
+-------+----------+------------+------------------+--------------+--------------------+--------------------+--------+---------+--------------------+-------------+--------+-------------------+
|      1|2021-03-21| 86010199872|218.49116722954264|    2021-03-21|  Orci In Foundation|computer programm...|       c|     2.40|Technology & Prof...|          2.4|    NULL|0.09805431136520959|
|      1|2021-03-21| 72472909171| 26.84427554025195|    2021-03-21|   Nullam Consulting|digital goods: bo...|       a|     6.33|Arts, Media & Ent...|         6.33|    NULL|0.09805431136520959|
|      1|2021-09-07| 82065156333| 7

### Tier C: Model to impute remaining probabilities (soft-label regression)


### Tier C Features


In [24]:
# Pre-training feature significance checks
# - Pearson correlation vs p_direct for numeric features

from pyspark.sql import functions as F

# Build minimal feature base if not present
if 'base' not in locals():
    if 'joined_b' in locals():
        ds = joined_b
    elif 'joined_a' in locals():
        ds = joined_a
    else:
        raise ValueError("Run Tier A/B cells first to create joined_a/joined_b before significance checks.")
    base = ds.withColumn("log_amount", F.log1p(F.col("dollar_value"))) \
             .withColumn("dow", F.dayofweek("order_date")) \
             .withColumn("month", F.month("order_date"))
    w_user_90 = (
        Window.partitionBy("user_id").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
    )
    base = base.withColumn("user_txn_count_90d", F.count(F.lit(1)).over(w_user_90)) \
               .withColumn("user_sum_90d", F.sum("dollar_value").over(w_user_90)) \
               .withColumn("user_avg_amount_90d", F.avg("dollar_value").over(w_user_90))
    w_user_prev = Window.partitionBy("user_id").orderBy("order_date")
    base = base.withColumn("prev_date", F.lag("order_date").over(w_user_prev)) \
               .withColumn("user_days_since_prev", F.datediff("order_date", "prev_date")) \
               .drop("prev_date")
    w_merch_90 = (
        Window.partitionBy("merchant_abn").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
    )
    base = base.withColumn("m_txn_count_90d", F.count(F.lit(1)).over(w_merch_90)) \
               .withColumn("m_sum_90d", F.sum("dollar_value").over(w_merch_90)) \
               .withColumn("m_avg_amount_90d", F.avg("dollar_value").over(w_merch_90))
    numeric_fill = [
        "log_amount", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
        "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
    ]
    base = base.fillna(0, subset=numeric_fill)

# Use labeled rows (where p_direct is available)
if 'labeled' not in locals():
    labeled = base.filter(F.col("p_direct").isNotNull()).cache()

print("Labeled rows for significance checks:", labeled.count())

numeric_features = [
    "log_amount",
    "dow",
    "month",
    "user_txn_count_90d",
    "user_sum_90d",
    "user_avg_amount_90d",
    "user_days_since_prev",
    "m_txn_count_90d",
    "m_sum_90d",
    "m_avg_amount_90d",
    "take_rate_num",
]

# Pearson correlations
corr_results = []
for feat in numeric_features:
    try:
        c = labeled.stat.corr(feat, "p_direct")
    except Exception:
        c = None
    corr_results.append((feat, c))

# Print correlations sorted by absolute value
corr_results_sorted = sorted(
    [(f, c) for f, c in corr_results if c is not None], key=lambda x: abs(x[1]), reverse=True
)
print("Pearson correlation with p_direct (top):")
for f, c in corr_results_sorted:
    print(f"  {f}: {c:.6f}")


Labeled rows for significance checks: 54764
Pearson correlation with p_direct (top):
  user_sum_90d: 0.412204
  m_avg_amount_90d: 0.252069
  user_avg_amount_90d: 0.182340
  log_amount: 0.069945
  take_rate_num: -0.012187
  user_txn_count_90d: 0.011514
  m_sum_90d: 0.009042
  month: -0.007889
  dow: -0.005769
  m_txn_count_90d: -0.004313
  user_days_since_prev: -0.003000


In [25]:
# Feature engineering for modeling (no income)

# Base for modeling
base = joined_b.withColumn(
    "p_label", F.coalesce(F.col("p_direct"), F.lit(None))
)

# Transaction-level
base = base.withColumn("log_amount", F.log1p(F.col("dollar_value"))) \
           .withColumn("dow", F.dayofweek("order_date")) \
           .withColumn("month", F.month("order_date"))

# User windows (90d)
w_user_90 = (
    Window.partitionBy("user_id").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
)
base = base.withColumn("user_txn_count_90d", F.count(F.lit(1)).over(w_user_90)) \
           .withColumn("user_sum_90d", F.sum("dollar_value").over(w_user_90)) \
           .withColumn("user_avg_amount_90d", F.avg("dollar_value").over(w_user_90))

# Days since previous txn per user
w_user_prev = Window.partitionBy("user_id").orderBy("order_date")
base = base.withColumn("prev_date", F.lag("order_date").over(w_user_prev)) \
           .withColumn("user_days_since_prev", F.datediff("order_date", "prev_date")) \
           .drop("prev_date")

# Merchant windows (90d)
w_merch_90 = (
    Window.partitionBy("merchant_abn").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
)
base = base.withColumn("m_txn_count_90d", F.count(F.lit(1)).over(w_merch_90)) \
           .withColumn("m_sum_90d", F.sum("dollar_value").over(w_merch_90)) \
           .withColumn("m_avg_amount_90d", F.avg("dollar_value").over(w_merch_90))

# Impute missing numeric features to avoid NaN/Inf in vectors
numeric_fill = [
    "log_amount", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
    "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
]
base = base.fillna(0, subset=numeric_fill)

# Categorical encodings: biz_tags, rev_band
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

indexers = [
    StringIndexer(inputCol="rev_band", outputCol="rev_band_idx", handleInvalid="keep"),
    StringIndexer(inputCol="biz_tags", outputCol="biz_tags_idx", handleInvalid="keep"),
]
encoders = [
    OneHotEncoder(inputCols=["rev_band_idx", "biz_tags_idx"], outputCols=["rev_band_oh", "biz_tags_oh"])
]

feats = [
    "log_amount", "dow", "month", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
    "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
]

assembler = VectorAssembler(inputCols=feats + ["rev_band_oh", "biz_tags_oh"], outputCol="features", handleInvalid="keep")

prep_pipeline = Pipeline(stages=indexers + encoders + [assembler])

# Labeled training data: where p_direct is available
labeled = base.filter(F.col("p_direct").isNotNull()).cache()
print("Labeled rows:", labeled.count())


Labeled rows: 54764


In [26]:
# Ablation study: retrain without income features (same split/seed)
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# Prepare ablation dataset by removing income features
indexers_no_income = [
    StringIndexer(inputCol="rev_band", outputCol="rev_band_idx", handleInvalid="keep"),
    StringIndexer(inputCol="biz_tags", outputCol="biz_tags_idx", handleInvalid="keep"),
]
encoders_no_income = [
    OneHotEncoder(inputCols=["rev_band_idx", "biz_tags_idx"], outputCols=["rev_band_oh", "biz_tags_oh"])
]

feats_no_income = [
    "log_amount", "dow", "month", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
    "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
]

assembler_no_income = VectorAssembler(inputCols=feats_no_income + ["rev_band_oh", "biz_tags_oh"], outputCol="features", handleInvalid="keep")

prep_pipeline_no_income = Pipeline(stages=indexers_no_income + encoders_no_income + [assembler_no_income])

prepared_no_income = prep_pipeline_no_income.fit(labeled).transform(labeled)
train_df_no_income, valid_df_no_income = prepared_no_income.randomSplit([0.8, 0.2], seed=SEED)

# Define models
lr_no_income = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.5, regParam=0.1)
gbt_no_income = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=6, maxIter=60, stepSize=0.1, subsamplingRate=0.8)

# Train
lr_model_no_income = lr_no_income.fit(train_df_no_income)
gbt_model_no_income = gbt_no_income.fit(train_df_no_income)

# Predict
pred_lr_no_income = lr_model_no_income.transform(valid_df_no_income)
pred_gbt_no_income = gbt_model_no_income.transform(valid_df_no_income)

# Clip to [0,1]
def _clip01_df(df, labelCol="p_direct", predCol="prediction"):
    return df.withColumn("lbl", F.when(F.col(labelCol) < 0, 0.0).when(F.col(labelCol) > 1, 1.0).otherwise(F.col(labelCol))) \
             .withColumn("prd", F.when(F.col(predCol) < 0, 0.0).when(F.col(predCol) > 1, 1.0).otherwise(F.col(predCol)))

mae_eval = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mae")
mse_eval = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mse")

cl_lr = _clip01_df(pred_lr_no_income)
cl_gbt = _clip01_df(pred_gbt_no_income)

metrics_ablate = {
    "lr_mae": mae_eval.evaluate(cl_lr),
    "lr_brier": mse_eval.evaluate(cl_lr),
    "gbt_mae": mae_eval.evaluate(cl_gbt),
    "gbt_brier": mse_eval.evaluate(cl_gbt),
}
print("Ablation metrics (no income):", metrics_ablate)

# If ablation is worse than original, try simpler GBT / stronger LR
try:
    baseline_metrics = metrics  # from previous training cell
except NameError:
    baseline_metrics = None

if baseline_metrics is None or (metrics_ablate["gbt_mae"] > baseline_metrics.get("gbt_mae", 1e9)):
    print("Ablation worse than baseline; simplifying models...")
    lr_tuned = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.7, regParam=0.3)
    gbt_tuned = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=4, maxIter=40, stepSize=0.08, subsamplingRate=0.8)

    lr_model_tuned = lr_tuned.fit(train_df_no_income)
    gbt_model_tuned = gbt_tuned.fit(train_df_no_income)

    pred_lr_tuned = lr_model_tuned.transform(valid_df_no_income)
    pred_gbt_tuned = gbt_model_tuned.transform(valid_df_no_income)

    cl_lr_tuned = _clip01_df(pred_lr_tuned)
    cl_gbt_tuned = _clip01_df(pred_gbt_tuned)

    metrics_tuned = {
        "lr_mae": mae_eval.evaluate(cl_lr_tuned),
        "lr_brier": mse_eval.evaluate(cl_lr_tuned),
        "gbt_mae": mae_eval.evaluate(cl_gbt_tuned),
        "gbt_brier": mse_eval.evaluate(cl_gbt_tuned),
    }
    print("Tuned metrics (no income):", metrics_tuned)



25/10/09 21:09:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/10/09 21:09:14 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Ablation metrics (no income): {'lr_mae': 0.05751549742653283, 'lr_brier': 0.007877208216177788, 'gbt_mae': 0.045380627575807714, 'gbt_brier': 0.005726630251325476}
Ablation worse than baseline; simplifying models...


Tuned metrics (no income): {'lr_mae': 0.05751549742653283, 'lr_brier': 0.007877208216177788, 'gbt_mae': 0.04539191641109387, 'gbt_brier': 0.005665177584323905}


In [27]:
# Train LR and GBT; evaluate and calibrate

from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

prepared = prep_pipeline.fit(labeled).transform(labeled)

train_df, valid_df = prepared.randomSplit([0.8, 0.2], seed=SEED)

lr = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.5, regParam=0.1)
gbt = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=6, maxIter=60, stepSize=0.1, subsamplingRate=0.8)

lr_model = lr.fit(train_df)
gbt_model = gbt.fit(train_df)

pred_lr = lr_model.transform(valid_df)
pred_gbt = gbt_model.transform(valid_df)

# Evaluate MAE and Brier (MSE on [0,1])
mae_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mae")
mse_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mse")

def _clip01(col):
    return F.when(F.col(col) < 0, 0.0).when(F.col(col) > 1, 1.0).otherwise(F.col(col))

def evaluate(df, label="p_direct", pred="prediction"):
    tmp = df.withColumn("lbl", _clip01(label)).withColumn("prd", _clip01(pred))
    mae   = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mae").evaluate(tmp)
    brier = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mse").evaluate(tmp)
    return mae, brier

metrics = {
    "lr_mae": evaluate(pred_lr)[0],
    "lr_brier": evaluate(pred_lr)[1],
    "gbt_mae": evaluate(pred_gbt)[0],
    "gbt_brier": evaluate(pred_gbt)[1],
}
print(metrics)

# Choose best model (skip isotonic calibration for speed)
best_is_gbt = metrics["gbt_mae"] <= metrics["lr_mae"]
best_model = gbt_model if best_is_gbt else lr_model


25/10/09 21:18:21 WARN BlockManager: Asked to remove block broadcast_2619, which does not exist


{'lr_mae': 0.05751549742653283, 'lr_brier': 0.007877208216177788, 'gbt_mae': 0.04538062757580773, 'gbt_brier': 0.005726630251325476}


In [28]:
# Apply best model to unlabeled rows and calibrate

prep_model = prep_pipeline.fit(labeled)
prepared = prep_model.transform(labeled)

# Re-train both on prepared
train_df, valid_df = prepared.randomSplit([0.8, 0.2], seed=SEED)

lr = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.5, regParam=0.1)
gbt = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=6, maxIter=60, stepSize=0.1, subsamplingRate=0.8)

lr_model = lr.fit(train_df)
gbt_model = gbt.fit(train_df)

pred_lr = lr_model.transform(valid_df)
pred_gbt = gbt_model.transform(valid_df)

mae_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mae")
mse_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mse")

metrics = {
    "lr_mae": mae_eval.evaluate(pred_lr),
    "lr_brier": mse_eval.evaluate(pred_lr),
    "gbt_mae": mae_eval.evaluate(pred_gbt),
    "gbt_brier": mse_eval.evaluate(pred_gbt),
}
print("Validation metrics:", metrics)

best_is_gbt = metrics["gbt_mae"] <= metrics["lr_mae"]
best_model = gbt_model if best_is_gbt else lr_model
best_valid = pred_gbt if best_is_gbt else pred_lr

unlabeled = base.filter(F.col("p_direct").isNull() & F.col("p_user_decay").isNull())
prepared_unlabeled = prep_model.transform(unlabeled)

scored_unlabeled = best_model.transform(prepared_unlabeled)

# Clip predictions to [0,1] as calibrated scores
scored_unlabeled = scored_unlabeled.withColumn(
    "p_model_calibrated",
    F.when(F.col("prediction") < 0, 0.0).when(F.col("prediction") > 1, 1.0).otherwise(F.col("prediction"))
)

print("Scored unlabeled rows:", scored_unlabeled.count())


Validation metrics: {'lr_mae': 0.05751549742653283, 'lr_brier': 0.007877208216177788, 'gbt_mae': 0.04538082875997335, 'gbt_brier': 0.005726670637477029}
Scored unlabeled rows: 2182835


In [29]:
# Coalesce probabilities and persist per-transaction output

# Bring together tiers
coalesced = (
    joined_b
    .join(scored_unlabeled.select("merchant_abn", "user_id", "order_date", "p_model_calibrated"), ["merchant_abn", "user_id", "order_date"], "left")
    .withColumn("p_hat", F.coalesce(F.col("p_direct"), F.col("p_user_decay"), F.col("p_model_calibrated")))
)

# Clip to [0,1], fallback to global mean if any remain null
mu = coalesced.select(F.mean("p_hat")).first()[0]
coalesced = coalesced.withColumn("p_hat", F.when(F.col("p_hat").isNull(), F.lit(mu)).otherwise(F.col("p_hat")))
coalesced = coalesced.withColumn("p_hat", F.when(F.col("p_hat") < 0, 0.0).when(F.col("p_hat") > 1, 1.0).otherwise(F.col("p_hat")))

per_tx_out = coalesced.select(
    "merchant_abn", "user_id", F.col("order_date").alias("order_datetime"), "dollar_value", "biz_tags", "rev_band", "take_rate", "segment", "p_hat"
)

print("Per-transaction output rows:", per_tx_out.count())


Per-transaction output rows: 13304011


In [30]:
from pyspark.sql import functions as F

# Tiers on joined_b: T1 if p_direct present; T2 if decay present; else T3
t1 = joined_b.filter(F.col("p_direct").isNotNull()).count()
t2 = joined_b.filter(F.col("p_direct").isNull() &
                     F.col("p_user_decay").isNotNull()).count()
t3 = joined_b.filter(F.col("p_direct").isNull() &
                     F.col("p_user_decay").isNull()).count()

total = joined_b.count()

print(f"Tier 1 (p_direct): {t1} ({t1/total:.2%})")
print(f"Tier 2 (user-decay): {t2} ({t2/total:.2%})")
print(f"Tier 3 (model): {t3} ({t3/total:.2%})")
print("Sanity check:", t1 + t2 + t3 == total)

Tier 1 (p_direct): 54764 (0.41%)
Tier 2 (user-decay): 11056382 (83.17%)
Tier 3 (model): 2182835 (16.42%)
Sanity check: True


### Merchant-level aggregation & Empirical-Bayes shrinkage


In [31]:
# Aggregate to merchant metrics with full scope and EB shrinkage

# Global mean from per-transaction outputs
mu = per_tx_out.select(F.avg("p_hat").alias("mu")).first()[0]

# Merchant scope (includes merchants with zero rows after filters)
merchants_scope = (
    spark.read.parquet(MERCHANTS_PATH)
    .select("merchant_abn")
    .distinct()
)

# Aggregate over the filtered/deduped transaction-level data you’ll rank on
agg_tx = (
    per_tx_out.groupBy("merchant_abn")
    .agg(
        F.count(F.lit(1)).alias("n_txn"),
        F.sum("dollar_value").alias("sum_amount"),
        F.avg("p_hat").alias("mean_p"),
        F.sum(F.col("p_hat") * F.col("dollar_value")).alias("EFL"),
    )
)

# Left-join to include zero-row merchants, fill sensible defaults
agg_complete = (
    merchants_scope.join(agg_tx, "merchant_abn", "left")
    .withColumn("n_txn", F.coalesce(F.col("n_txn"), F.lit(0)))
    .withColumn("sum_amount", F.coalesce(F.col("sum_amount"), F.lit(0.0)))
    .withColumn("EFL", F.coalesce(F.col("EFL"), F.lit(0.0)))
)

# EFLR = 0 when sum_amount == 0; otherwise EFL / sum_amount
agg_complete = agg_complete.withColumn(
    "EFLR",
    F.when(F.col("sum_amount") == 0, F.lit(0.0)).otherwise(F.col("EFL") / F.col("sum_amount"))
)

# EB shrinkage: if n_txn == 0, set eb_p to mu by convention
agg_complete = agg_complete.withColumn(
    "eb_p",
    F.when(F.col("n_txn") == 0, F.lit(mu)).otherwise(
        (F.col("n_txn") * F.col("mean_p") + F.lit(EB_KAPPA) * F.lit(mu)) /
        (F.col("n_txn") + F.lit(EB_KAPPA))
    )
)

# Optional: SE of mean_p; set 0 when n_txn == 0
agg_complete = agg_complete.withColumn(
    "se_mean_p",
    F.when(F.col("n_txn") == 0, F.lit(0.0)).otherwise(
        F.sqrt(F.col("mean_p") * (1 - F.col("mean_p")) / F.greatest(F.col("n_txn"), F.lit(1)))
    )
)


In [32]:
# Rankings: Best-100 (safest to onboard) and Worst-100 (investigate)


merchants_sel = (
    spark.read.parquet(MERCHANTS_PATH)
    .select("merchant_abn", F.col("name").alias("merchant_name"))
)

agg_named = agg_complete.join(F.broadcast(merchants_sel), "merchant_abn", "left")

best_100 = (
    agg_named
    .orderBy(F.col("eb_p").asc())
    .select("merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
    .limit(100)
)
worst_100 = (
    agg_named
    .orderBy(F.col("EFL").desc())
    .select("merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
    .limit(100)
)

print("Best-100 (safest to onboard):")
best_100.show(20, truncate=False)
print("Worst-100 (investigate):")
worst_100.show(20, truncate=False)

# Save to curated folder
best_100.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/best_100_merchants.csv")
worst_100.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/worst_100_merchants.csv")

print("Saved rankings to curated folder")

Best-100 (safest to onboard):


+------------+------------------------------+-----+------------------+-------------------+------------------+-------------------+
|merchant_abn|merchant_name                 |n_txn|sum_amount        |eb_p               |EFL               |EFLR               |
+------------+------------------------------+-----+------------------+-------------------+------------------+-------------------+
|82044452251 |Malesuada Integer Id Company  |1064 |1347129.1570628267|0.14656699623287983|189700.85788345375|0.1408186118523798 |
|23109950638 |Etiam Bibendum Fermentum LLC  |3700 |926601.1182960935 |0.14749769508288657|135236.66722453572|0.1459491733327707 |
|85556646149 |Proin Nisl Inc.               |1956 |584194.1682930886 |0.14766571822153693|85190.91177947611 |0.1458263645944786 |
|60829135130 |Tellus Imperdiet Non Inc.     |1123 |2815206.2030041288|0.14772224185038701|407796.28729536687|0.14485485534246273|
|66161087361 |Morbi LLC                     |1337 |853339.9183561503 |0.14778060283759764|

+------------+------------------------------+------+-----------------+-------------------+------------------+-------------------+
|merchant_abn|merchant_name                 |n_txn |sum_amount       |eb_p               |EFL               |EFLR               |
+------------+------------------------------+------+-----------------+-------------------+------------------+-------------------+
|19492220327 |Commodo Ipsum Industries      |825   |8203379.006304084|0.19108541550264566|2134997.8501913724|0.2602583458049033 |
|90918180829 |Pharetra Quisque Company      |562   |5683135.042278277|0.19214190916402643|1627139.4399822073|0.28631018405818376|
|96680767841 |Ornare Limited                |31119 |9773037.435149513|0.15214450889315528|1483874.7156920242|0.15183352417695134|
|27093785141 |Placerat Orci Institute       |25921 |9717646.139408296|0.1523375385814382 |1477736.1678125905|0.15206729557890336|
|50315283629 |Iaculis Aliquet Diam LLC      |29834 |9600351.709830722|0.1545526534995159 |

Saved rankings to curated folder


In [33]:
agg_named.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/merchant_fraud_rankings.csv")

In [34]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Compute dominant segment per merchant (by transaction count)
per_tx_seg = (
    per_tx_out
    .where(F.col("merchant_abn").isNotNull() & F.col("segment").isNotNull())
    .select("merchant_abn", "segment")
)

seg_counts = per_tx_seg.groupBy("merchant_abn", "segment").count()

w = Window.partitionBy("merchant_abn").orderBy(F.col("count").desc(), F.col("segment").asc())
dominant_seg = seg_counts.withColumn("rn", F.row_number().over(w)).where(F.col("rn") == 1).drop("rn")

# Attach dominant segment to merchant aggregation
agg_with_seg = agg_complete.join(F.broadcast(dominant_seg), "merchant_abn", "left")

# Save a complete table with dominant segment
agg_with_seg_named = agg_with_seg.join(F.broadcast(merchants_sel), "merchant_abn", "left")
agg_with_seg_named.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/merchant_fraud_rankings_with_segment.csv")
print("Saved merchant_fraud_rankings_with_segment.csv")


Saved merchant_fraud_rankings_with_segment.csv


In [ ]:
from pyspark.sql import functions as F

# Best merchants per segment
# Criterion: lowest eb_p within each segment, with tie-breakers by higher n_txn and lower EFL

# If a merchant has no dominant segment (null), label as 'Unknown'
agg_with_seg_named = agg_with_seg_named.fillna({"segment": "Unknown"})

w_seg = (
    Window.partitionBy("segment")
    .orderBy(F.col("eb_p").asc(), F.col("n_txn").desc(), F.col("EFL").asc(), F.col("merchant_abn").asc())
)
ranked_by_segment = (
    agg_with_seg_named
    .withColumn("seg_rank", F.row_number().over(w_seg))
)

# Top N per segment (N=100 by default, easy to change)
TOP_N = 10

best_by_segment = (
    ranked_by_segment
    .where(F.col("seg_rank") <= TOP_N)
    .select("segment", "merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
    .orderBy(F.col("segment").asc(), F.col("seg_rank").asc())
)

best_by_segment.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/best_merchants_by_segment.csv")
print("Saved best_merchants_by_segment.csv with top merchants per segment")



Saved best_merchants_by_segment.csv with top merchants per segment


In [36]:
best_by_segment.show(20, truncate=False)

+---------------------------------+------------+---------------------------------+-----+------------------+-------------------+------------------+-------------------+
|segment                          |merchant_abn|merchant_name                    |n_txn|sum_amount        |eb_p               |EFL               |EFLR               |
+---------------------------------+------------+---------------------------------+-----+------------------+-------------------+------------------+-------------------+
|Arts, Media & Entertainment      |43801715289 |Mollis Duis Sit Foundation       |414  |560035.8975183531 |0.1480279458387124 |82580.41218571554 |0.14745556945840116|
|Arts, Media & Entertainment      |46145508777 |Fringilla Est Industries         |406  |311261.6220841651 |0.14806920944713384|43410.982800374055|0.1394678293767798 |
|Arts, Media & Entertainment      |55646417663 |Nisi Cum LLC                     |243  |144626.44903045817|0.14843316173005536|19716.821746978545|0.13632929439362923